In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, f1_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.base import clone

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
from experiment_runner import load_dataset
from imputation import imputation

np.random.seed(42)

In [7]:
datasets = ["housing", "adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

# глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

ratios = [0.05, 0.20, 0.50]
mechanisms = ["MCAR", "MAR"]

# число повторов для каждой комбинации (dataset x mechanism x method x ml_model x ratio)
N_REPEATS = 5
BASE_RANDOM_STATE = 42

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
    ],
    "KNN hybrid": [
        {"n_neighbors": 7},
    ],
    "MICE hybrid": [
        {"max_iter": 10},
    ],
    "MissForest": [
        {"n_estimators": 100},
    ]
}

In [8]:
# функция для вычисления метрик моделей
def evaluate_model(y_true, y_pred, task_type="regression"):
    if task_type == "regression":
        rmse = root_mean_squared_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        return {"rmse": rmse, "r2": r2}
    else:
        accuracy = (y_true == y_pred).mean()
        f1 = f1_score(y_true, y_pred, average="weighted")
        metrics = {"accuracy": accuracy, "f1": f1}
        return metrics

# словари с моделями для разных задач
models = {
    "housing": {
        "task_type": "regression",
        "models": {
            "Ridge": Ridge(random_state=42),
            "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        }
    },
    "adult": {
        "task_type": "classification",
        "models": {
            "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
            "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        }
    }
}


In [9]:
results_list = []  # список для сбора результатов в таблицу

for dataset_name in datasets:
    # загружаем данные и таргет
    df_true = load_dataset(dataset_name)
    target_col = global_target[dataset_name]

    task_type = models[dataset_name]["task_type"]
    current_models = models[dataset_name]["models"]

    print(f"\n[{dataset_name.upper()}] Task: {task_type}")

    # сохраняем таргет
    y_true_target = df_true[target_col]
    # удаляем таргет из признаков, чтобы методы импутации не видели его
    X_true_features_raw = df_true.drop(columns=[target_col])

    # ohe
    X_true_features = pd.get_dummies(X_true_features_raw, drop_first=True)

    # перебираем механизмы пропусков
    for mechanism in mechanisms:
        # кэш пропусков: одинаковые шаблоны для всех методов и моделей
        missing_data_cache = {}
        for ratio in ratios:
            for repeat_id in range(N_REPEATS):
                repeat_seed = BASE_RANDOM_STATE + repeat_id
                df_incomplete, mask = generate_missing_data(
                    data=X_true_features_raw,
                    columns_config=columns_target[dataset_name],
                    mechanism=mechanism,
                    ratio=ratio,
                    random_state=repeat_seed
                )
                missing_data_cache[(ratio, repeat_id)] = (df_incomplete, mask)
            print(
                f"Missing data generated for {dataset_name} | {mechanism} | Ratio: {ratio} | repeats: {N_REPEATS}"
            )

        for method, param_list in EXPERIMENT_CONFIG.items():
            for params in param_list:
                params_str = "_".join([f"{k}={v}" for k, v in params.items()])

                for ml_model_name, ml_model in current_models.items():

                    exp_name = (
                        f"{dataset_name} | {mechanism} | {method} | ML: {ml_model_name} | repeats={N_REPEATS}"
                    )

                    # baseline (ratio=0): усредняем по повторам (разные split seed)
                    baseline_values = {}
                    for repeat_id in range(N_REPEATS):
                        split_seed = BASE_RANDOM_STATE + repeat_id
                        X_tr_true, X_te_true, y_tr_true, y_te_true = train_test_split(
                            X_true_features, y_true_target, test_size=0.2, random_state=split_seed
                        )

                        model_instance = clone(ml_model)
                        model_instance.fit(X_tr_true, y_tr_true)
                        y_pred_base = model_instance.predict(X_te_true)
                        metrics_base = evaluate_model(y_te_true, y_pred_base, task_type)

                        for met_name, met_val in metrics_base.items():
                            baseline_values.setdefault(met_name, []).append(met_val)

                    baseline_mean = {k: float(np.mean(v)) for k, v in baseline_values.items()}
                    baseline_std = {
                        f"{k}_std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
                        for k, v in baseline_values.items()
                    }

                    base_res = {
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "method": method,
                        "params": params_str,
                        "ml_model": ml_model_name,
                        "ratio": 0.0,
                        "n_repeats": N_REPEATS
                    }

                    base_res.update(baseline_mean)
                    base_res.update(baseline_std)
                    for met_name, met_val in baseline_mean.items():
                        base_res[f"{met_name}_mean"] = met_val
                    results_list.append(base_res)

                    # ratio > 0: прогоняем несколько повторов и усредняем
                    for ratio in ratios:
                        print(
                            f"[{dataset_name}] {mechanism} | {method} | {ml_model_name} | Ratio: {ratio} | repeats={N_REPEATS}"
                        )

                        ratio_metric_values = {}

                        for repeat_id in range(N_REPEATS):
                            split_seed = BASE_RANDOM_STATE + repeat_id
                            df_incomplete, mask = missing_data_cache[(ratio, repeat_id)]

                            # восстанавливаем
                            df_imputed = imputation(
                                df_incomplete=df_incomplete,
                                algo=method,
                                **params
                            )

                            # подготовка данных
                            X_imputed = pd.get_dummies(df_imputed, drop_first=True)
                            X_imputed = X_imputed.reindex(columns=X_true_features.columns, fill_value=0)

                            # разделяем
                            X_train, X_test, y_train, y_test = train_test_split(
                                X_imputed, y_true_target, test_size=0.2, random_state=split_seed
                            )

                            # обучение и предсказание
                            model_instance = clone(ml_model)
                            model_instance.fit(X_train, y_train)
                            y_pred = model_instance.predict(X_test)

                            # оценка
                            metrics = evaluate_model(y_test, y_pred, task_type)

                            for met_name, met_val in metrics.items():
                                ratio_metric_values.setdefault(met_name, []).append(met_val)

                        ratio_mean = {k: float(np.mean(v)) for k, v in ratio_metric_values.items()}
                        ratio_std = {
                            f"{k}_std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
                            for k, v in ratio_metric_values.items()
                        }

                        step = int(ratio * 100)

                        ratio_res = {
                            "dataset": dataset_name,
                            "mechanism": mechanism,
                            "method": method,
                            "params": params_str,
                            "ml_model": ml_model_name,
                            "ratio": ratio,
                            "n_repeats": N_REPEATS
                        }
                        ratio_res.update(ratio_mean)
                        ratio_res.update(ratio_std)
                        for met_name, met_val in ratio_mean.items():
                            ratio_res[f"{met_name}_mean"] = met_val
                        results_list.append(ratio_res)

                    print(f"-> Finished logging all ratios for {exp_name}")

# сохраняем результаты в Excel
df_results = pd.DataFrame(results_list)
df_results.to_excel("model_evaluation_results.xlsx", index=False)


[HOUSING] Task: regression
Missing data generated for housing | MCAR | Ratio: 0.05 | repeats: 5
Missing data generated for housing | MCAR | Ratio: 0.2 | repeats: 5
Missing data generated for housing | MCAR | Ratio: 0.5 | repeats: 5
[housing] MCAR | Simple | Ridge | Ratio: 0.05 | repeats=5
[housing] MCAR | Simple | Ridge | Ratio: 0.2 | repeats=5
[housing] MCAR | Simple | Ridge | Ratio: 0.5 | repeats=5
-> Finished logging all ratios for housing | MCAR | Simple | ML: Ridge | repeats=5
[housing] MCAR | Simple | RandomForest | Ratio: 0.05 | repeats=5
[housing] MCAR | Simple | RandomForest | Ratio: 0.2 | repeats=5
[housing] MCAR | Simple | RandomForest | Ratio: 0.5 | repeats=5
-> Finished logging all ratios for housing | MCAR | Simple | ML: RandomForest | repeats=5
[housing] MCAR | KNN hybrid | Ridge | Ratio: 0.05 | repeats=5
[housing] MCAR | KNN hybrid | Ridge | Ratio: 0.2 | repeats=5
[housing] MCAR | KNN hybrid | Ridge | Ratio: 0.5 | repeats=5
-> Finished logging all ratios for housing | 

/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100

[adult] MCAR | Simple | LogisticRegression | Ratio: 0.05 | repeats=5


/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100

[adult] MCAR | Simple | LogisticRegression | Ratio: 0.2 | repeats=5


...

[adult] MAR | MissForest | LogisticRegression | Ratio: 0.2 | repeats=5


/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100

[adult] MAR | MissForest | LogisticRegression | Ratio: 0.5 | repeats=5


/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100

-> Finished logging all ratios for adult | MAR | MissForest | ML: LogisticRegression | repeats=5
[adult] MAR | MissForest | RandomForest | Ratio: 0.05 | repeats=5
[adult] MAR | MissForest | RandomForest | Ratio: 0.2 | repeats=5
[adult] MAR | MissForest | RandomForest | Ratio: 0.5 | repeats=5
-> Finished logging all ratios for adult | MAR | MissForest | ML: RandomForest | repeats=5
